# RouteHunter

In [1]:
import os
import pandas as pd

from routehunter import RouteHunterApp

In [3]:
app_data_dir = "rh_data"
app = RouteHunterApp.from_data_dir(app_data_dir)
print(app.load_report.summary())

[09:36:22] Invalid InChI prefix in generating InChI Key


Rows processed : 1481
Targets        : 1481
Unique targets : 1362
Unique papers  : 1263
Errors         : 0


### 1. Review

In [4]:
print(app.review())

RouteHunter: A system for the collection and distribution of reference information on chemical synthesis routes

  Search      : give a SMILES, get papers, static CASP-solved tool
                results, and predicted solvability for that molecule.
                
  Predict     : given a SMILES, get predicted solvability probability
                per CASP tool, with a link to that tool.
                
  Monitor     : browse recently published papers, ranked by predicted
                probability of containing a multi-step synthesis route
                (pre-scored offline; candidates for you to review and
                add to the CSV by hand).


RouteHunter data review:
  Targets                    : 1362
  Papers                     : 1263
  Targets with >1 paper      : 97
  Cached CASP routes         : 0
  Predicted candidate papers : 122865 (awaiting for digitalization)
  Papers by journal:
    Organic Process Research & Development   1240
    European Journal of Organic 

### 2. Search

Given a SMILES, return literature papers *and* any CASP-predicted routes cached earlier this session.

Try some molecules with positive search:  
``C#CCOC1=C(C=C(C(=C1)N2C(=O)N3CCCCC3=N2)Cl)Cl``  
``C(O)(C(O)=O)C(C1C=CC=CC=1)NC(C1C=CC=CC=1)=O``  
``C1CCC(=C(C1)CC(=O)O)N2C(=O)C=CC(=N2)C3=C4C=CC=CN4N=C3C5=CC=CC=C5``

Try some absent molecules:  
``CC(C)Cc1ccc(cc1)C(C)C(=O)O``

In [5]:
result = app.search("C(O)(C(O)=O)C(C1C=CC=CC=1)NC(C1C=CC=CC=1)=O")
print(result.report())

Found 1 paper(s) reporting a route for this molecule:
 - [paper] Utilization of a Benzoyl Migration To Effect an Expeditious Synthesis of the Paclitaxel C-13 Side Chain (Organic Process Research & Development, 1997) doi:10.1021/op970113b

Found 2 tool(s) predicted routes for this molecule:
 - [AiZynthFinder] This molecule was solved by AiZynthFinder. See predicted routes: Cached predicted routes are not available yet.
 - [SynPlanner] This molecule was solved by SynPlanner. See predicted routes: Cached predicted routes are not available yet.


## 3. Predict

Predict a route computationally. Results are cached in memory for this session (`cache=True` by default) so a later Search this session surfaces them too — but the cache disappears when the notebook restarts; it is never written to the CSV. Uses a stub `CASPEngine` here — swap in a real open-source CASP tool (e.g. AiZynthFinder) via the same `predict_route` interface.

In [6]:
result = app.predict("C(O)(C(O)=O)C(C1C=CC=CC=1)NC(C1C=CC=CC=1)=O")
print(result.to_dataframe().to_string())

            tool probability                                                             url
0  AiZynthFinder         84%                    https://github.com/MolecularAI/aizynthfinder
1     SynPlanner         82%  https://github.com/Laboratoire-de-Chemoinformatique/SynPlanner


In [7]:
result.to_dataframe()

,tool,probability,url
0,AiZynthFinder,84%,https://github.com/MolecularAI/aizynthfinder
1,SynPlanner,82%,https://github.com/Laboratoire-de-Chemoinforma...


### 4. Monitor

Fetch recent papers, score with a classifier, display ranked. This is **display-only** - nothing here is written into the dataset. If a candidate turns out to be a real route, the way to record it is to add a row to the CSV and reload.

In [8]:
result  = app.monitor(year_min=1990, year_max=2025)
print(result.message)

16878 paper(s) for 1990-2025, sorted by predicted route probability.


In [9]:
result.to_dataframe()

,route_prob,journal,title,publication_date,doi
0,83%,Organic Process Research & Development,Practical Synthesis of a HIV Integrase Inhibitor,29/10/2008,10.1021/op800153y
1,82%,Tetrahedron,An expeditious route to the synthesis of adeno...,01/05/1996,10.1016/0040-4039(96)00632-6
2,81%,Organic Process Research & Development,Development of a Scalable Route to the SMO Rec...,31/10/2012,10.1021/op300170q
3,81%,Tetrahedron,An efficient route for synthesis of spirocycli...,14/08/2024,10.1016/j.tetlet.2024.155250
4,81%,Tetrahedron,Synthesis of combretastatin D-2. An efficient ...,01/06/1994,10.1016/s0040-4039(00)73369-7
...,...,...,...,...,...
16873,58%,Synlett,Extending the Utility of the Bartoli Indolizat...,23/01/2013,10.1055/s-0032-1318137
16874,58%,Angewandte Chemie International Edition,Catalytic Asymmetric Total Synthesis of <i>ent...,08/01/2010,10.1002/anie.200906678
16875,58%,Journal of Organic Chemistry,Modular and Stereodivergent Approach to Unbran...,26/08/2016,10.1021/acs.joc.6b01051
16876,58%,Organic Letters,Asymmetric Total Synthesis of (−)-Spirofungin ...,09/11/2005,10.1021/ol052039k


### 5. Download

Export the dataset (or a filtered slice) as a flat table for ML training. Every row comes from the CSV — Hunter output never appears here since it's never written into the dataset.

In [10]:
config_df = pd.read_csv(os.path.join(app_data_dir, "config.csv"))
config_df

,key,path,comment
0,TargetStaticData,static/target_static_data.csv,Digitalized collection of targets
1,AizynthfinderStaticData,static/aizynthfinder_static_data.csv,AiZynthFinder solved-by-tool table
2,SynplannerStaticData,static/synplanner_static_data.csv,SynPlanner solved-by-tool table
3,MonitorStaticData,static/monitor_static_data.csv,High-confidence paper with route candidates
4,CandidateStaticData,static/candidate_static_data.csv,Medium-confidence paper with route candidates
5,AbstractTrainingData,build/abstract_training_data.csv,Training data for paper classifier
6,AizynthfinderPredictModel,model/aizynthfinder_predict_model.pickle,AiZynthFinder solvability model
7,SynplannerPredictModel,model/synplanner_predict_model.pickle,SynPlanner solvability model
8,PaperPredictModel,model/paper_predict_model.pickle,Paper-with-route classifier model
